In [10]:
from dotenv import load_dotenv
import os
import time

load_dotenv(dotenv_path=".env")

SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
TENANT_ID = os.environ["AZURE_TENANT_ID"]
os.environ["SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["TENANT_ID"] = TENANT_ID
os.environ["WORKDIR"] = "/tmp/vault"
os.environ["RESOURCE_GROUP"] = "VaultDemoRG"
os.environ["APP_NAME"] = "vaultsecretsync-mapfre"
# Azure Key Vault names are globally unique and at most 24 characters.
os.environ["KEYVAULT"] = f"kvmapfre{int(time.time())}"


Requires https://docs.prod.secops.hashicorp.services/doormat/azure/working_with_ad/

In [11]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')

# Log into Azure CLI

In [12]:
! az login --tenant $TENANT_ID --subscription $SUBSCRIPTION_ID

A web browser has been opened at https://login.microsoftonline.com/237fbc04-c52a-458b-af97-eaf7157c0cd4/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.

Retrieving subscriptions for the selection...
[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "237fbc04-c52a-458b-af97-eaf7157c0cd4",
    "id": "581bc96c-4f27-4978-8b49-c3a0729ffb2e",
    "isDefault": true,
    "managedByTenants": [],
    "name": "mapfre-test",
    "state": "Enabled",
    "tenantId": "237fbc04-c52a-458b-af97-eaf7157c0cd4",
    "user": {
      "name": "jose.merchan@hashicorp.services",
      "type": "user"
    }
  }
]


In [13]:
%%bash
az account show \
  --query '{subscription:name, subscriptionId:id, tenantId:tenantId}' \
  --output table

Subscription    SubscriptionId                        TenantId
--------------  ------------------------------------  ------------------------------------
mapfre-test     581bc96c-4f27-4978-8b49-c3a0729ffb2e  237fbc04-c52a-458b-af97-eaf7157c0cd4


In [14]:
%%bash
az provider register \
  --namespace Microsoft.KeyVault \
  --subscription "$SUBSCRIPTION_ID"

In [15]:
! az provider show \
  --namespace Microsoft.KeyVault \
  --subscription "$SUBSCRIPTION_ID" \
  --query '{namespace:namespace,state:registrationState}' \
  --output table

Namespace           State
------------------  ----------
Microsoft.KeyVault  Registered


### Create Resource Group

In [16]:
! az group create --name $RESOURCE_GROUP --location westeurope 

{
  "id": "/subscriptions/581bc96c-4f27-4978-8b49-c3a0729ffb2e/resourceGroups/VaultDemoRG",
  "location": "westeurope",
  "managedBy": null,
  "name": "VaultDemoRG",
  "properties": {
    "provisioningState": "Succeeded"
  },
  "tags": null,
  "type": "Microsoft.Resources/resourceGroups"
}


In [17]:
! az group show --name $RESOURCE_GROUP | jq -r '.id'

/subscriptions/581bc96c-4f27-4978-8b49-c3a0729ffb2e/resourceGroups/VaultDemoRG


## Create KeyVault

In [18]:
%%bash
set -euo pipefail
if ! az keyvault show --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP" >/dev/null 2>&1; then
  az keyvault create --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP" --location westeurope >/dev/null
fi
az keyvault show --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP" \
  --query '{name:name,id:id,rbac:properties.enableRbacAuthorization}' -o json

{
  "id": "/subscriptions/581bc96c-4f27-4978-8b49-c3a0729ffb2e/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/kvmapfre1790166776",
  "name": "kvmapfre1790166776",
  "rbac": true
}


### Create App, SP, Role Assignment and Credentials

In [ ]:
# to delete the app registration
# az ad app delete --id 100cb49d-f583-42b7-ab60-56f89dc0338b

In [20]:
%%bash
set -euo pipefail
echo "Create or reuse App Registration"
export APP_ID=$(az ad app list --display-name "$APP_NAME" --query '[0].appId' -o tsv)
if [[ -z "$APP_ID" ]]; then
  APP_ID=$(az ad app create --display-name "$APP_NAME" --query appId -o tsv)
fi
echo $APP_ID

echo "Create Service Principal"
az ad sp show --id "$APP_ID" >/dev/null 2>&1 || az ad sp create --id "$APP_ID" >/dev/null

Create or reuse App Registration
79d505bf-ce92-493f-88f2-7f0e261b8748
Create Service Principal


## Assign permisions to app

In [21]:
%%bash
CLIENT_ID=$(az ad app list --display-name $APP_NAME --query "[0].appId" -o tsv)
az role assignment create \
  --assignee $CLIENT_ID \
  --role "Key Vault Administrator" \
  --scope "/subscriptions/$SUBSCRIPTION_ID/resourceGroups/$RESOURCE_GROUP/providers/Microsoft.KeyVault/vaults/$KEYVAULT" || true

{
  "condition": null,
  "conditionVersion": null,
  "createdBy": null,
  "createdOn": "2026-09-23T12:36:18.674663+00:00",
  "delegatedManagedIdentityResourceId": null,
  "description": null,
  "id": "/subscriptions/581bc96c-4f27-4978-8b49-c3a0729ffb2e/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/kvmapfre1790166776/providers/Microsoft.Authorization/roleAssignments/63e8f3e1-14ec-400b-82ec-4bb7ff07a00c",
  "name": "63e8f3e1-14ec-400b-82ec-4bb7ff07a00c",
  "principalId": "a0d069c7-2810-436b-bf0e-54648a94bb77",
  "principalType": "ServicePrincipal",
  "resourceGroup": "VaultDemoRG",
  "roleDefinitionId": "/subscriptions/581bc96c-4f27-4978-8b49-c3a0729ffb2e/providers/Microsoft.Authorization/roleDefinitions/00482a5a-887f-4fb3-b363-3b7fe8e74483",
  "scope": "/subscriptions/581bc96c-4f27-4978-8b49-c3a0729ffb2e/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/kvmapfre1790166776",
  "systemData": null,
  "type": "Microsoft.Authorization/roleAssignments",
  "update

## Configure Vault to connect

In [22]:
%%bash
set -euo pipefail

CLIENT_ID=$(
  az ad app list \
    --display-name "$APP_NAME" \
    --query "[0].appId" \
    --output tsv
)

CLIENT_SECRET=$(
  az ad app credential reset \
    --id "$CLIENT_ID" \
    --display-name "vault-secrets-sync" \
    --append \
    --years 1 \
    --query "password" \
    --output tsv \
    --only-show-errors |
  tr -d '\r'
)

VAULT_URI=$(
  az keyvault show \
    --name "$KEYVAULT" \
    --resource-group "$RESOURCE_GROUP" \
    --query "properties.vaultUri" \
    --output tsv
)

if [[ -z "$CLIENT_ID" || -z "$CLIENT_SECRET" || -z "$VAULT_URI" ]]; then
  echo "A required Azure value is empty" >&2
  exit 1
fi

sleep 50

vault write sys/sync/destinations/azure-kv/azure-sync \
  key_vault_uri="$VAULT_URI" \
  client_id="$CLIENT_ID" \
  client_secret="$CLIENT_SECRET" \
  tenant_id="$TENANT_ID" \
  secret_name_template='vault-{{ .MountPath | replace "/" "-" | replace "_" "-" | lowercase }}-{{ .SecretPath | replace "/" "-" | replace "_" "-" | lowercase }}'

if ! vault secrets list -format=json | jq -e 'has("sync-azure-cli/")' >/dev/null; then
  vault secrets enable -path=sync-azure-cli kv-v2
fi
vault kv put sync-azure-cli/verification \
  scenario=azure-cli-spn \
  message='Managed independently by 5_Secret_Sync_Azure_CLI_SPN.ipynb'
vault write sys/sync/destinations/azure-kv/azure-sync/associations/set \
  mount=sync-azure-cli \
  secret_name=verification


unset CLIENT_SECRET

Key                   Value
---                   -----
connection_details    map[client_id:79d505bf-ce92-493f-88f2-7f0e261b8748 client_secret:***** key_vault_uri:https://kvmapfre1790166776.vault.azure.net/ tenant_id:237fbc04-c52a-458b-af97-eaf7157c0cd4]
name                  azure-sync
options               map[custom_tags:map[] granularity_level:secret-path secret_name_template:vault_sync_{{ .SecretBaseName | lowercase }}]
type                  azure-kv
Success! Enabled the kv-v2 secrets engine at: sync-azure-cli/
========== Secret Path ==========
sync-azure-cli/data/verification

======= Metadata =======
Key                Value
---                -----
created_time       2026-09-23T12:41:12.831869734Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            1
Key                        Value
---                        -----
associated_secrets         map[kv_309c7f65/verification:map[accessor:kv_309c7f65 external_name:vault-sync-verification last_o

## Add permissions to login user for access to the Key Vault

In [23]:
%%bash
set -euo pipefail
ACCOUNT_TYPE=$(az account show --query user.type -o tsv)
if [[ "${ACCOUNT_TYPE}" == "user" ]]; then
  PRINCIPAL_ID=$(az ad signed-in-user show --query id -o tsv)
  PRINCIPAL_TYPE=User
else
  CLIENT_ID=$(az account show --query user.name -o tsv)
  PRINCIPAL_ID=$(az ad sp show --id "${CLIENT_ID}" --query id -o tsv)
  PRINCIPAL_TYPE=ServicePrincipal
fi
KEYVAULT_ID=$(az keyvault show --name "${KEYVAULT}" --resource-group "${RESOURCE_GROUP}" --query id -o tsv)
if ! az role assignment list --assignee "${PRINCIPAL_ID}" --scope "${KEYVAULT_ID}" \
  --query "[?roleDefinitionName=='Key Vault Administrator'] | [0].id" -o tsv | grep -q .; then
  az role assignment create --assignee-object-id "${PRINCIPAL_ID}" \
    --assignee-principal-type "${PRINCIPAL_TYPE}" \
    --role "Key Vault Administrator" --scope "${KEYVAULT_ID}" >/dev/null
fi
for attempt in $(seq 1 30); do
  az keyvault secret list --vault-name "${KEYVAULT}" --maxresults 1 >/dev/null 2>&1 && break
  [[ "${attempt}" -eq 30 ]] && { echo 'Azure RBAC no se propagó a tiempo' >&2; exit 1; }
  sleep 10
done

In [24]:
%%bash
az keyvault secret list --vault-name $KEYVAULT --query "[].{Name:name, Value:value}" -o table

Name
-----------------------
vault-sync-verification


# Read the sync secret

In [25]:
! # Read the Azure sync destination
! vault read sys/sync/destinations/azure-kv/azure-sync

Key                   Value
---                   -----
connection_details    map[client_id:79d505bf-ce92-493f-88f2-7f0e261b8748 client_secret:***** key_vault_uri:https://kvmapfre1790166776.vault.azure.net/ tenant_id:237fbc04-c52a-458b-af97-eaf7157c0cd4]
name                  azure-sync
options               map[granularity_level:secret-path secret_name_template:vault_sync_{{ .SecretBaseName | lowercase }}]
type                  azure-kv


In [26]:
!# Read all associations and their sync status
!vault read sys/sync/destinations/azure-kv/azure-sync/associations

Key                        Value
---                        -----
associated_secrets         map[kv_309c7f65/verification:map[accessor:kv_309c7f65 external_name:vault-sync-verification last_operation:Write mount:sync-azure-cli secret_name:verification sync_status:SYNCED updated_at:2026-09-23T12:41:13.369965061Z]]
store_name                 azure-sync
store_type                 azure-kv
sync_operation_counters    map[SYNCED:1]


In [27]:
%%bash
az keyvault secret show \
  --vault-name "$KEYVAULT" \
  --name "vault-sync-azure-cli-verification" \
  --query '{name:name, enabled:attributes.enabled, updated:attributes.updated}' \
  --output json

{
  "enabled": true,
  "name": "vault-sync-verification",
  "updated": "2026-09-23T12:41:13+00:00"
}


In [28]:
%%bash
az keyvault secret show \
  --vault-name "$KEYVAULT" \
  --name "vault-sync-azure-cli-verification" \
  --query value \
  --output tsv

{"message":"Managed independently by 5_Secret_Sync_Azure_CLI_SPN.ipynb","scenario":"azure-cli-spn"}


# CLEAN UP

Run this only when you want to remove the PoC. Vault is cleaned first so it can unsync external secrets while the SPN and Key Vault still exist.

In [29]:
%%bash
set -euo pipefail

DESTINATION_PATH="sys/sync/destinations/azure-kv/azure-sync"

echo "Current Vault associations"
vault read -format=json "$DESTINATION_PATH/associations" | jq '.data.associated_secrets // {}' || true

echo "Purging the Vault destination and unsyncing associated secrets"
if vault read "$DESTINATION_PATH" >/dev/null 2>&1; then
  vault delete "$DESTINATION_PATH" purge=true

  destination_deleted=false
  for attempt in {1..24}; do
    if ! vault read "$DESTINATION_PATH" >/dev/null 2>&1; then
      destination_deleted=true
      echo "Vault destination deleted"
      break
    fi
    echo "Waiting for Vault to finish the purge (${attempt}/24)..."
    sleep 5
  done

  if [[ "$destination_deleted" != "true" ]]; then
    echo "Vault destination still exists. Azure resources were not deleted." >&2
    echo "Inspect the association status before considering force_delete." >&2
    exit 1
  fi
else
  echo "Vault destination does not exist; continuing"
fi
if vault secrets list -format=json | jq -e 'has("sync-azure-cli/")' >/dev/null; then
  vault secrets disable sync-azure-cli/
fi

CLIENT_ID=$(az ad app list --display-name "$APP_NAME" --query '[0].appId' --output tsv)

if [[ -n "$CLIENT_ID" ]]; then
  echo "Deleting the service principal"
  az ad sp delete --id "$CLIENT_ID" || true

  echo "Deleting the app registration"
  az ad app delete --id "$CLIENT_ID" || true
else
  echo "App registration does not exist; continuing"
fi

if az keyvault show --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP" >/dev/null 2>&1; then
  echo "Deleting Azure Key Vault"
  az keyvault delete --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP"
fi

if az keyvault show-deleted --name "$KEYVAULT" >/dev/null 2>&1; then
  echo "Purging the soft-deleted Azure Key Vault"
  az keyvault purge --name "$KEYVAULT" --location westeurope
fi

if az group exists --name "$RESOURCE_GROUP" | grep -qx true; then
  echo "Deleting the resource group"
  az group delete --name "$RESOURCE_GROUP" --yes
fi

echo "Cleanup completed"

Current Vault associations
{
  "kv_309c7f65/verification": {
    "accessor": "kv_309c7f65",
    "external_name": "vault-sync-verification",
    "last_operation": "Write",
    "mount": "sync-azure-cli",
    "secret_name": "verification",
    "sync_status": "SYNCED",
    "updated_at": "2026-09-23T12:41:13.369965061Z"
  }
}
Purging the Vault destination and unsyncing associated secrets
Success! Data deleted (if it existed) at: sys/sync/destinations/azure-kv/azure-sync
Waiting for Vault to finish the purge (1/24)...
Vault destination deleted
Success! Disabled the secrets engine (if it existed) at: sync-azure-cli/
Deleting the service principal
Deleting the app registration
Deleting Azure Key Vault


https://learn.microsoft.com/azure/key-vault/general/soft-delete-overview


Purging the soft-deleted Azure Key Vault
Deleting the resource group
Cleanup completed
